# Exploring Colombian Spanish Corpora with `open_corpus_co_es`

**Course:** Natural Language Processing · Unit 2 — Statistical Patterns in Language
**Institution:** Universidad Tecnológica de Bolívar
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 2 — Words and Tokens
**Library:** [`open_corpus_co_es`](https://pypi.org/project/open_corpus_co_es/) 0.1.1a4 (L. G. Moreno Sandoval, Pontificia Universidad Javeriana)
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Browse a **corpus catalog** and pick datasets by type, topic, country and size.
2. Download and load corpora reproducibly (checksums, local cache).
3. Measure basic corpus statistics: **tokens, types, type–token ratio**.
4. Check **Zipf's law** and **Heaps' law** on real Spanish text.
5. Compare **vocabulary across countries** (Colombia · Perú · Ecuador news).
6. Apply an **affective lexicon** to presidential speeches.

> Files are downloaded from Google Drive into `~/.open_corpus_co_es/data/<name>/` the first time; later runs reuse the local copy.

In [ ]:
# Pre-release version: the pin is required (pip ignores alpha releases otherwise)
!pip install --user -U "open_corpus_co_es==0.1.1a4"

---

## 1. Environment Setup

In [ ]:
# Standard library
import re
from collections import Counter
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.text import Text

from open_corpus_co_es import download_corpus, list_corpus, load_corpus
from open_corpus_co_es.downloader import CATALOG, get_corpus_path

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
STOP_ES = set(stopwords.words('spanish'))

pd.set_option('display.max_colwidth', 80)
print("Ready.")

---

## 2. The Catalog

`list_corpus()` returns a dictionary `{name: metadata}` with the **enabled** entries only.
We turn it into a DataFrame to filter and sort.

In [ ]:
inventario = list_corpus()

catalogo = (
    pd.DataFrame.from_dict(inventario, orient='index')
      .assign(MB=lambda d: (d['bytes'].fillna(0) / 1e6).round(1),
              filas=lambda d: d['filas'].fillna(d['documentos']))
      [['tipo', 'origen', 'tema', 'pais', 'año', 'extension', 'filas', 'MB', 'descripcion']]
      .sort_values(['tipo', 'origen', 'tema'])
)
print(f"Enabled entries: {len(catalogo)}")
catalogo.head(10)

In [ ]:
# How is the catalog distributed?
resumen = (catalogo.groupby(['tipo', 'origen'])
                   .agg(entradas=('tema', 'size'), filas=('filas', 'sum'), MB=('MB', 'sum'))
                   .sort_values('MB', ascending=False))
resumen

In [ ]:
# Example filters: small Colombian corpora (< 5 MB) — good for quick classroom experiments
catalogo.query("tipo == 'corpora' and pais == 'Colombia' and MB < 5")[['tema', 'filas', 'MB', 'descripcion']]

### 2.1 Labeled corpora for supervised tasks

Several entries carry a **label column**, so they can feed later units (classification, sequence models, transformers):

| Corpus | Text column | Label(s) | Rows | Possible task |
|---|---|---|---|---|
| `es_hs_project_raw_extended_v2` | `text` | `HS`, `TR`, `AG` | 4,950 | Hate speech (HatEval-style) |
| `discurso_odio` | `text` | `HS`, `RS`, `MS` | 915,939 | Hate speech, large scale |
| `llm_humano` | `Texto` | `Clasificacion` | 1,844 | Human vs. LLM-generated text |
| `edades_entrenamiento` | `text` | `age_range` | 169,336 | Author profiling (age) |
| `news_2020_*_v2` | `text` | `topic` | 17k–27k | Topic classification |
| `salud_/educacion_/seguridad_*_2024` | `Hit Sentence` | `Sentiment` | 20,000 each | Sentiment (social media) |

---

## 3. Loading Helper

`load_corpus()` converts every row into a `dict` (`{"text": ..., **metadata}`). That is convenient for small corpora but slow for
large ones (e.g. `discurso_odio`, 916k rows). The helper below downloads with the library (checksum-validated) and then
reads the file **directly with pandas**.

In [ ]:
def cargar_df(nombre: str, force: bool = False) -> pd.DataFrame:
    """Download (if needed) a catalog entry and return it as a DataFrame."""
    download_corpus(nombre, force=force)
    meta = CATALOG[nombre]
    ext = meta['extension'].lower()
    ruta = Path(get_corpus_path(nombre)) / f"{nombre}.{ext}"
    if ext == 'parquet':
        return pd.read_parquet(ruta)
    if ext == 'xlsx':
        return pd.read_excel(ruta)
    if ext == 'csv':
        return pd.read_csv(ruta, sep=None, engine='python')
    raise ValueError(f"Unsupported extension: {ext}")


TOKEN_RE = re.compile(r"[a-záéíóúüñ]+", re.IGNORECASE)

def tokenizar(texto: str) -> list[str]:
    """Lower-cased alphabetic tokens (Spanish accents and ñ included)."""
    return TOKEN_RE.findall(str(texto).lower())

---

## 4. Case Study 1 — Colombian Presidential Speeches (`presidentes`)

Same steps as the library's quick-start.

In [ ]:
nombre = "presidentes"
download_corpus(nombre)
documentos = load_corpus(nombre)

print(len(documentos))
print(documentos[0]["text"][:200])
print("Metadata keys:", [k for k in documentos[0] if k != 'text'])

In [ ]:
pres = pd.DataFrame(documentos)
pres['tokens'] = pres['text'].map(tokenizar)
pres['n_tokens'] = pres['tokens'].map(len)
pres['n_types'] = pres['tokens'].map(lambda t: len(set(t)))
pres['TTR'] = (pres['n_types'] / pres['n_tokens']).round(3)

pres[['nombre', 'fecha_inicio', 'fecha_fin', 'n_tokens', 'n_types', 'TTR']].sort_values('fecha_inicio')

> **Question:** TTR falls as texts get longer, so comparing raw TTR across speeches of different length is misleading.
> A fairer comparison uses a fixed window (e.g. the first 1,000 tokens), as in the next cell.

In [ ]:
VENTANA = 1000
pres['TTR_1000'] = pres['tokens'].map(lambda t: len(set(t[:VENTANA])) / VENTANA if len(t) >= VENTANA else np.nan)
pres[['nombre', 'n_tokens', 'TTR', 'TTR_1000']].sort_values('TTR_1000', ascending=False)

In [ ]:
# Most frequent content words across all speeches
frec = Counter(w for toks in pres['tokens'] for w in toks if w not in STOP_ES and len(w) > 2)
pd.DataFrame(frec.most_common(25), columns=['palabra', 'frecuencia'])

In [ ]:
# Keyword in context with NLTK
texto_nltk = Text(w for toks in pres['tokens'] for w in toks)
texto_nltk.concordance('paz', width=90, lines=12)

### 4.1 Zipf's Law

Zipf: the frequency of a word is roughly inversely proportional to its rank, $f(r) \propto r^{-\alpha}$ with $\alpha \approx 1$.
On a log–log plot this is a straight line with slope $-\alpha$.

In [ ]:
def zipf(tokens, titulo):
    freqs = np.array(sorted(Counter(tokens).values(), reverse=True))
    rangos = np.arange(1, len(freqs) + 1)
    alpha, c = np.polyfit(np.log(rangos), np.log(freqs), 1)
    plt.figure(figsize=(6, 4))
    plt.loglog(rangos, freqs, '.', ms=3, label='observed')
    plt.loglog(rangos, np.exp(c) * rangos ** alpha, '-', label=f'fit  slope = {alpha:.2f}')
    plt.xlabel('rank'); plt.ylabel('frequency'); plt.title(titulo); plt.legend(); plt.show()
    return alpha

todos = [w for toks in pres['tokens'] for w in toks]
zipf(todos, 'Zipf — presidential speeches');

---

## 5. Case Study 2 — News from Colombia, Perú and Ecuador (2020)

Three comparable corpora (`_id`, `topic`, `text`), ~40–50 MB each. Download may take a minute.

In [ ]:
PAISES = {'Colombia': 'news_2020_colombia_v2', 'Perú': 'news_2020_peru_v2', 'Ecuador': 'news_2020_ecuador_v2'}

noticias = pd.concat(
    [cargar_df(n).assign(pais=p) for p, n in PAISES.items()],
    ignore_index=True,
)
noticias['tokens'] = noticias['text'].map(tokenizar)
noticias['n_tokens'] = noticias['tokens'].map(len)

noticias.groupby('pais').agg(documentos=('text', 'size'),
                             tokens=('n_tokens', 'sum'),
                             tokens_por_doc=('n_tokens', 'median'))

In [ ]:
# Topic distribution per country (top 10 topics overall)
top_topics = noticias['topic'].value_counts().head(10).index
pd.crosstab(noticias.loc[noticias.topic.isin(top_topics), 'topic'],
            noticias.loc[noticias.topic.isin(top_topics), 'pais'], normalize='columns').round(3)

### 5.1 Heaps' Law

Vocabulary size grows sub-linearly with corpus size: $V(N) = k N^{\beta}$, typically $0.4 < \beta < 0.7$.

In [ ]:
def curva_heaps(tokens, pasos=50):
    vistos, puntos = set(), []
    paso = max(1, len(tokens) // pasos)
    for i, w in enumerate(tokens, 1):
        vistos.add(w)
        if i % paso == 0:
            puntos.append((i, len(vistos)))
    return np.array(puntos)

plt.figure(figsize=(6, 4))
for pais, grupo in noticias.groupby('pais'):
    toks = [w for t in grupo['tokens'] for w in t][:3_000_000]   # same budget for every country
    pts = curva_heaps(toks)
    beta, logk = np.polyfit(np.log(pts[:, 0]), np.log(pts[:, 1]), 1)
    plt.plot(pts[:, 0], pts[:, 1], label=f'{pais}  β = {beta:.2f}')
plt.xlabel('tokens (N)'); plt.ylabel('vocabulary (V)'); plt.title("Heaps' law — news 2020"); plt.legend(); plt.show()

### 5.2 Distinctive Vocabulary per Country

Log-odds ratio with an informative Dirichlet prior (Monroe, Colaresi & Quinn, 2008), as presented in Jurafsky & Martin.
Positive z-scores: words over-represented in that country relative to the other two.

In [ ]:
def log_odds_z(c_a: Counter, c_b: Counter, prior: Counter, a0: float = 0.01):
    n_a, n_b = sum(c_a.values()), sum(c_b.values())
    alpha = {w: a0 * f for w, f in prior.items()}
    a_sum = sum(alpha.values())
    out = {}
    for w, aw in alpha.items():
        ya, yb = c_a.get(w, 0), c_b.get(w, 0)
        d = np.log((ya + aw) / (n_a + a_sum - ya - aw)) - np.log((yb + aw) / (n_b + a_sum - yb - aw))
        var = 1 / (ya + aw) + 1 / (yb + aw)
        out[w] = d / np.sqrt(var)
    return pd.Series(out)

conteos = {p: Counter(w for t in g['tokens'] for w in t if w not in STOP_ES and len(w) > 2)
           for p, g in noticias.groupby('pais')}
prior = sum(conteos.values(), Counter())

distintivas = {}
for pais in conteos:
    resto = sum((c for p, c in conteos.items() if p != pais), Counter())
    distintivas[pais] = log_odds_z(conteos[pais], resto, prior).nlargest(15).index.tolist()
pd.DataFrame(distintivas)

---

## 6. Case Study 3 — Affective Lexicon on Presidential Speeches

`lexicon_afectivo_categorias` maps Spanish words to SEL and NRC emotion categories.
We compute, for each speech, the share of tokens in each NRC category.

In [ ]:
lex = cargar_df('lexicon_afectivo_categorias')
print(lex.shape)
print(lex['category_Ncr'].value_counts().head(12))
lex.head()

In [ ]:
lex_ncr = (lex.dropna(subset=['category_Ncr'])
              .assign(Palabra=lambda d: d['Palabra'].str.lower().str.strip())
              .drop_duplicates(['Palabra', 'category_Ncr']))
palabra2cats = lex_ncr.groupby('Palabra')['category_Ncr'].apply(set).to_dict()

def perfil_afectivo(tokens):
    c = Counter(cat for w in tokens for cat in palabra2cats.get(w, ()))
    return {cat: 1000 * n / max(len(tokens), 1) for cat, n in c.items()}   # per 1,000 tokens

perfiles = pd.DataFrame(pres['tokens'].map(perfil_afectivo).tolist(), index=pres['nombre']).fillna(0).round(2)
perfiles.sort_index(axis=1)

In [ ]:
# Heat map: emotion categories per 1,000 tokens
fig, ax = plt.subplots(figsize=(9, 0.35 * len(perfiles) + 1))
im = ax.imshow(perfiles.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(perfiles.shape[1]), perfiles.columns, rotation=45, ha='right')
ax.set_yticks(range(perfiles.shape[0]), perfiles.index)
fig.colorbar(im, ax=ax, label='per 1,000 tokens'); ax.set_title('NRC affect profile — presidential speeches')
plt.tight_layout(); plt.show()

> **Caveat:** lexicon methods ignore negation, irony and word sense (*"no hay **paz**"* still counts as positive). Treat these profiles as descriptive, not as sentiment classification.

---

## 7. Exercises

1. **Catalog:** Which three corpora would you use for a *Colombia vs. México vs. Argentina* comparison of social-media posts about health? Justify with rows, dates and text column.
2. **TTR:** Compute `TTR_1000` for the news corpora by country. Is the ranking stable if you change the window to 500 or 2,000 tokens?
3. **Zipf:** Fit Zipf's slope for each country's news. Does removing stop-words change $\alpha$? Why?
4. **Distinctive words:** Repeat §5.2 per `topic` instead of per country. Which topics have the most country-specific vocabulary?
5. **Supervised task (preview of Unit 3):** Load `es_hs_project_raw_extended_v2` with `cargar_df` and train a TF-IDF + Logistic Regression baseline for the `HS` label. Report macro-F1 on a stratified 80/20 split.

---

## References

- Jurafsky, D., & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed. draft), Ch. 2.
- Monroe, B. L., Colaresi, M. P., & Quinn, K. M. (2008). Fightin' words: Lexical feature selection and evaluation for identifying the content of political conflict. *Political Analysis, 16*(4), 372–403.
- Mohammad, S. M., & Turney, P. D. (2013). Crowdsourcing a word–emotion association lexicon. *Computational Intelligence, 29*(3), 436–465.
- Moreno Sandoval, L. G. *open_corpus_co_es* (v0.1.1a4) [Software]. PyPI.